In [24]:
import config
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_cerebras import ChatCerebras
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import ConversationalRetrievalChain
# Updated imports for modern LangChain
# Updated imports for LangChain 2026
from langchain_classic.chains import (
    create_history_aware_retriever, 
    create_retrieval_chain
)
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

In [2]:
url = "https://365datascience.com/upcoming‐courses"

In [3]:
loader = WebBaseLoader(url)

In [4]:
raw_documents = loader.load()

In [6]:
text_splitter = RecursiveCharacterTextSplitter()
documents = text_splitter.split_documents(raw_documents)

In [7]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [13]:
ids = [f"chunk_{i}" for i in range(len(documents))]

# Pass the ids list directly to FAISS
vectorstore = FAISS.from_documents(
    documents=documents,
    embedding=embeddings,
    ids=ids
)

In [14]:
from langchain.memory import ConversationBufferMemory

In [16]:
memory = ConversationBufferMemory(
    memory_key = 'chat_history',
    return_messages = True
)

In [27]:
llm = ChatCerebras(
    model='llama3.1-8b',
    api_key = config.apikey,
    temperature = 0
)

In [31]:
# 1. Define the system prompt for the answer chain
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, try to find it. If not found say you don't know and could not find anything "
    "Use three sentences maximum and keep the answer concise.\n\n"
    "{context}"
)

# 2. Create the prompt templates
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

# 3. Create the document processing chain
# Note: This replaces the 'stuff' chain type from your original code
question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

# 4. Create the final RAG chain
rag_chain = create_retrieval_chain(vectorstore.as_retriever(), question_answer_chain)

# 5. Execute a query
# Note: History-aware chains use 'input' instead of 'question'
response = rag_chain.invoke({
    "input": "What are the upcoming courses?", 
    "chat_history": memory.load_memory_variables({})["chat_history"]
})

print(response["answer"])

I don't know and could not find anything about upcoming courses on the provided context.


In [32]:
response = rag_chain.invoke({
    "input": "what are the courses on the site?",
    "chat_history": memory.load_memory_variables({})["chat_history"]
})
print(response["answer"])

I don't know the courses on the site, the page is not found.
